## 1. Unicode Normalization

Unicode normalization converts text into a consistent Unicode representation. 
We use Python's unicodedata library to normalize the movie reviews.

In [5]:
import pandas as pd
import unicodedata

df = pd.read_csv("../data/IMDB Dataset.csv")

def normalize_unicode(text):
    return unicodedata.normalize("NFKC", text)

df["review_unicode"] = df["review"].apply(normalize_unicode)

print("Original Review:")
print(df["review"][0][:300])

print("\nNormalized Review:")
print(df["review_unicode"][0][:300])

Original Review:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Tru

Normalized Review:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Tru


In [6]:
changed_reviews = (df["review"] != df["review_unicode"]).sum()

print("Reviews changed after Unicode normalization:", changed_reviews)

Reviews changed after Unicode normalization: 223


### Observation

After applying NFKC Unicode normalization, 223 reviews out of 50,000 were changed (0.446%). The remaining 49,777 reviews remained unchanged. This shows that only a small portion of the dataset contained characters whose Unicode representation needed normalization.

## 2. Lowercasing

Lowercasing converts all uppercase letters into lowercase letters.
It helps maintain consistency by treating words with different capitalization
as the same word.

In [7]:
df["review_lower"] = df["review_unicode"].str.lower()

print("Original Review:")
print(df["review_unicode"][0][:300])

print("\nLowercase Review:")
print(df["review_lower"][0][:300])

Original Review:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Tru

Lowercase Review:
one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. they are right, as this is exactly what happened with me.<br /><br />the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. tru


In [8]:
changed_reviews = (df["review_unicode"] != df["review_lower"]).sum()

print("Reviews changed after lowercasing:", changed_reviews)

Reviews changed after lowercasing: 49780


### Observation

After applying lowercasing, 49,780 reviews out of 50,000 were changed (99.56%). The remaining 220 reviews (0.44%) remained unchanged. Lowercasing helps maintain consistency by treating words with different capitalization as the same word.

## 3. Noise Removal

Noise removal eliminates unnecessary elements from movie reviews,
such as HTML tags, URLs, unwanted special characters, and extra spaces.

We preserve apostrophes because contractions and negation words
can carry important sentiment information.

In [11]:
import re

def remove_noise(text):
    # Remove HTML tags
    text = re.sub(r"<[^>]+>", "", text)

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", "", text)

    # Remove unwanted special characters
    text = re.sub(r"[^\w\s'!?]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

df["review_clean_noise"] = df["review_lower"].apply(remove_noise)

print("Original Review:")
print(df["review_lower"][0][:300])

print("\nCleaned Review:")
print(df["review_clean_noise"][0][:300])


changed_reviews = (df["review_lower"] != df["review_clean_noise"]).sum()

print("Reviews changed after noise removal:", changed_reviews)

Original Review:
one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. they are right, as this is exactly what happened with me.<br /><br />the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. tru

Cleaned Review:
one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked they are right as this is exactly what happened with me the first thing that struck me about oz was its brutality and unflinching scenes of violence which set in right from the word go trust me this is no
Reviews changed after noise removal: 49987


In [12]:
import nltk
nltk.download("stopwords")

from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

print("Total Stopwords:", len(stop_words))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\aaari\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


Total Stopwords: 198


In [13]:
negation_words = {
    "no", "not", "nor", "don't", "didn't",
    "doesn't", "isn't", "wasn't", "weren't",
    "won't", "wouldn't", "couldn't", "shouldn't",
    "can't", "cannot"
}

stop_words = stop_words - negation_words

print("Stopwords after preserving negation:", len(stop_words))

Stopwords after preserving negation: 185


In [14]:
def remove_stopwords(text):
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words]
    return " ".join(filtered_words)

df["review_no_stopwords"] = df["review_clean_noise"].apply(remove_stopwords)

print("Original Review:")
print(df["review_clean_noise"][0][:300])

print("\nAfter Stopword Removal:")
print(df["review_no_stopwords"][0][:300])

Original Review:
one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked they are right as this is exactly what happened with me the first thing that struck me about oz was its brutality and unflinching scenes of violence which set in right from the word go trust me this is no

After Stopword Removal:
one reviewers mentioned watching 1 oz episode hooked right exactly happened first thing struck oz brutality unflinching scenes violence set right word go trust not show faint hearted timid show pulls no punches regards drugs sex violence hardcore classic use word called oz nickname given oswald maxi


### Observation

After stopword removal, common English words were removed to reduce unnecessary text. However, negation words such as "not" and "no" were preserved because they can change the sentiment of a movie review. This helps maintain important sentiment information.

In [ ]:
from nltk.stem import PorterStemmer, WordNetLemmatizer

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\aaari\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\aaari\AppData\Roaming\nltk_data...


True

In [16]:
words = ["playing", "studies", "connected", "better", "running"]

print("Stemming:")
for word in words:
    print(word, "->", stemmer.stem(word))

print("\nLemmatization:")
for word in words:
    print(word, "->", lemmatizer.lemmatize(word))

Stemming:
playing -> play
studies -> studi
connected -> connect
better -> better
running -> run

Lemmatization:
playing -> playing
studies -> study
connected -> connected
better -> better
running -> running


In [17]:
from nltk.corpus import wordnet

def lemmatize_word(word):
    return lemmatizer.lemmatize(word, wordnet.VERB)

print("Improved Lemmatization:")

for word in words:
    print(word, "->", lemmatize_word(word))

Improved Lemmatization:
playing -> play
studies -> study
connected -> connect
better -> better
running -> run


In [18]:
def lemmatize_text(text):
    words = text.split()
    lemmatized_words = [lemmatizer.lemmatize(word, wordnet.VERB) for word in words]
    return " ".join(lemmatized_words)

df["review_lemmatized"] = df["review_no_stopwords"].apply(lemmatize_text)

print("Before Lemmatization:")
print(df["review_no_stopwords"][0][:300])

print("\nAfter Lemmatization:")
print(df["review_lemmatized"][0][:300])

Before Lemmatization:
one reviewers mentioned watching 1 oz episode hooked right exactly happened first thing struck oz brutality unflinching scenes violence set right word go trust not show faint hearted timid show pulls no punches regards drugs sex violence hardcore classic use word called oz nickname given oswald maxi

After Lemmatization:
one reviewers mention watch 1 oz episode hook right exactly happen first thing strike oz brutality unflinching scenes violence set right word go trust not show faint hearted timid show pull no punch regard drug sex violence hardcore classic use word call oz nickname give oswald maximum security stat


## 6. Reusable Text Cleaning Function

A reusable text cleaning function combines all preprocessing steps into one function. It makes the cleaning process easier to apply to new movie reviews.

In [21]:
def clean_text(text):
    # Unicode normalization
    text = unicodedata.normalize("NFKC", text)

    # Lowercase
    text = text.lower()

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", "", text)

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", "", text)

    # Remove unwanted special characters
    text = re.sub(r"[^\w\s'!?]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Remove stopwords while preserving negation
    words = text.split()
    words = [word for word in words if word not in stop_words]

    # Lemmatization
    words = [lemmatizer.lemmatize(word, wordnet.VERB) for word in words]

    return " ".join(words)

sample = "This Movie was REALLY good! I didn't like the ending."

print(clean_text(sample))

movie really good! didn't like end


In [22]:
df["cleaned_review"] = df["review"].apply(clean_text)

print("Original Review:")
print(df["review"][0][:300])

print("\nCleaned Review:")
print(df["cleaned_review"][0][:300])

print("\nDataset Shape:", df.shape)

Original Review:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Tru

Cleaned Review:
one reviewers mention watch 1 oz episode hook right exactly happen first thing strike oz brutality unflinching scenes violence set right word go trust not show faint hearted timid show pull no punch regard drug sex violence hardcore classic use word call oz nickname give oswald maximum security stat

Dataset Shape: (50000, 8)


In [23]:
original_words = set()

for review in df["review"]:
    original_words.update(review.lower().split())

print("Vocabulary Before Cleaning:", len(original_words))



cleaned_words = set()

for review in df["cleaned_review"]:
    cleaned_words.update(review.split())

print("Vocabulary After Cleaning:", len(cleaned_words))


print("Vocabulary Before Cleaning:", len(original_words))
print("Vocabulary After Cleaning:", len(cleaned_words))

reduction = len(original_words) - len(cleaned_words)

print("Vocabulary Reduction:", reduction)

Vocabulary Before Cleaning: 390931
Vocabulary After Cleaning: 131728
Vocabulary Before Cleaning: 390931
Vocabulary After Cleaning: 131728
Vocabulary Reduction: 259203


### Observation

Before cleaning, the dataset contained 390,931 unique words. After applying text preprocessing, the vocabulary decreased to 131,728 unique words.

A total of 259,203 words were reduced, representing approximately 66.3% reduction. This shows that preprocessing removed unnecessary words and helped reduce vocabulary size.

In [24]:
df[["cleaned_review", "sentiment"]].to_csv(
    "../data/cleaned_imdb_dataset.csv",
    index=False
)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [25]:
cleaned_df = pd.read_csv("../data/cleaned_imdb_dataset.csv")

print(cleaned_df.head())
print("\nDataset Shape:", cleaned_df.shape)

                                      cleaned_review sentiment
0  one reviewers mention watch 1 oz episode hook ...  positive
1  wonderful little production film technique una...  positive
2  think wonderful way spend time hot summer week...  positive
3  basically there's family little boy jake think...  negative
4  petter mattei's love time money visually stun ...  positive

Dataset Shape: (50000, 2)
